# Macro-Factor Return Forecasting and Portfolio Optimization using the Black-Litterman Model

## Introduction

This notebook implements a quantitative portfolio optimization framework based on the Black–Litterman model, which extends the classical Markowitz approach by incorporating subjective investor views in a Bayesian setting.

Specifically, this notebook:

- Uses the PyPortfolioOpt library to implement the Black–Litterman model
(see documentation: https://pyportfolioopt.readthedocs.io/en/latest/BlackLitterman.html
)

- Integrates a multiple linear regression model to predict expected asset returns based on macroeconomic factors, which are then translated into Black–Litterman views

- Performs backtesting on real stock and ETF data to evaluate portfolio behavior

**Motivation**

The classical Markowitz mean–variance framework is highly sensitive to return estimates, often resulting in unstable and unintuitive portfolio allocations.
The Black–Litterman model addresses this issue by combining implied market equilibrium returns with customized views and confidence levels, leading to more robust and practically implementable portfolios.

**Libraries used**

- PyPortfolioOpt — portfolio optimization and Black–Litterman implementation

- yfinance — historical asset price data

- scikit-learn — linear regression models

- pandas_datareader — macroeconomic data

- matplotlib / seaborn — data visualization

In [ ]:
!pip install PyPortfolioOpt yfinance pandas numpy matplotlib seaborn scikit-learn pandas_datareader

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from pypfopt import BlackLittermanModel, EfficientFrontier, expected_returns, risk_models, black_litterman, plotting
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import pandas_datareader.data as web
from datetime import datetime

print("Imports done !")

## Theory and Reminders

### Markowitz Model (Modern Portfolio Theory)

The classical portfolio optimization framework is defined as follows:

- **Expected portfolio return**:  
  $$ R_p = w^\top \mu $$

- **Portfolio risk (variance)**:  
  $$ \sigma_p^2 = w^\top \Sigma w $$

- **Optimization problem (minimum variance)**:
  $$
  \min_w \; w^\top \Sigma w
  \quad \text{s.t.} \quad
  w^\top \mu = R_{\text{target}}, \quad
  w^\top \mathbf{1} = 1, \quad
  w \geq 0
  $$

Solving this problem for different target returns traces out the **efficient frontier**.

---

### Black–Litterman Model

The Black–Litterman model is a Bayesian extension of the Markowitz framework that incorporates **subjective investor views**.

- **Implied equilibrium returns (prior)**:
  $$
  \pi = \delta \Sigma w_m
  $$

  where:
  * δ is the risk aversion parameter
  * wₘ are the market capitalization weights

- **Investor views**:

  * P ∈ ℝ^(K×N): view matrix  
  * Q ∈ ℝ^K: view returns  
  * Ω ∈ ℝ^(K×K): diagonal covariance matrix of view uncertainty


- **Posterior expected returns**:
  $$
  \mu_{BL} =
  \left[(\tau \Sigma)^{-1} + P^\top \Omega^{-1} P\right]^{-1}
  \left[(\tau \Sigma)^{-1} \pi + P^\top \Omega^{-1} Q\right]
  $$

- **Posterior covariance (simplified form)**:
  $$
  \Sigma_{BL} = \Sigma + \left[(\tau \Sigma)^{-1} + P^\top \Omega^{-1} P\right]^{-1}
  $$
  with $\tau$ representing uncertainty on prior, usually $\in [0.01, 0.05]$

- **Going back to Markowitz now**
  * We go back to Markowitz optimization but using as input, the **BL** returns and covariance, the purpose being to **maximize** the ***Sharpe Ratio*** of the portfolio or to **minimize** its variance
  $$\underset {\omega}{max} \left( \omega^T \mu_{BL} - \frac {\delta} {2} \omega^T \Sigma_{BL} \omega\right)$$
  * We will use the pypfopt library for that

---

### Linear Regression for expected returns
For each asset $i$, we model the expected returns as:

$$
R_{i,t+1} = \alpha_i + \beta_{i,1} F^1_t + \beta_{i,2} F^2_t + \dots + \beta_{i,k} F^k_t + \varepsilon_{i,t+1}
$$

where:

- $R_{i,t+1}$ is the return of asset $i$ at time $t+1$,
- $F^j_t$ are the observed macroeconomic factors at time $t$,
- $\beta_{i,j}$ measure the sensitivity (exposure) of asset $i$ to factor $j$,
- $\alpha_i$ is the intercept (asset-specific alpha),
- $\varepsilon_{i,t+1}$ is an idiosyncratic error term assumed to be normally distributed.

### To visualize the Black-Litterman flow:

Market priors → Views (predicted via ML) → Adjusted returns → Optimization

## Data Collection and Preparation

For simplicity, we select 10 assets: NVDA, MSFT, JPM, JNJ, XOM, WMT, AAPL, PLTR, BAC and RTX. Period: 5 years.

In [ ]:
tickers = ["NVDA", "MSFT", "JPM", "JNJ", "XOM", "WMT", "AAPL", "PLTR", "BAC", "RTX"]

def DisplayFullName(tickers):
    tickerNames = {}

    for ticker in tickers: 
        stock = yf.Ticker(ticker)
        tickerNames[stock] = stock.info.get("longName")
    return tickerNames

In [ ]:
def getPrices(tickers,start_date="2015-01-01"):
    end_date = datetime.today().strftime('%Y-%m-%d')
    ohlc = yf.download(tickers, start=start_date, end=end_date, progress=False)
    prices = ohlc["Close"]
    return prices

In [ ]:
def DisplayMarketCaps(tickers):
    
    marketCaps = {}

    for ticker in tickers:
        stock = yf.Ticker(ticker)
        marketCaps[ticker] = stock.info.get("marketCap")

    totalMarketCap = sum(
        cap for cap in marketCaps.values() if cap is not None
    )

    weights = {
        ticker: cap / totalMarketCap if cap is not None else None
        for ticker, cap in marketCaps.items()
    }

    return weights, marketCaps

## Black-Litterman Implementation
Once we understand the theory behind the model, the true challenge is to evaluate correctly the different parameters of the model, which are :
- $\pi$ : The equilibrium prior that depends on 
    - $\delta$ : The risk aversion, **evaluated using Sharpe Ratio Method**
    - $\Sigma$ : The covariance matrix, evaluated using :
        - Sample covariance method (<span style="color:green">simple, non biased </span> but <span style="color:red">noisy on small samples, Singular if T \lt N </span>) $$\Sigma = \frac{1}{T-1} \sum_{t=1}^{T} (r_t - \bar{r})(r_t - \bar{r})^T$$
        
        - Semicovariance method (<span style="color:green">Captures only downside risk, Coherent with risk aversion </span> but <span style="color:red">Upside information is lost and statistically less stable </span>) $$\Sigma_{semi} = \frac{1}{T} \sum_{t: r_t < \bar{r}} (r_t - \bar{r})(r_t - \bar{r})^T$$
        
        - Exponential Covariance, that gives more weight to recent observations (<span style="color:green">reacts to regime changes </span> but $\lambda$ <span style="color:red"> choice is arbirary </span>) $$\Sigma_{exp} = \frac{\sum_{t} \lambda^{T-t} (r_t - \bar{r})(r_t - \bar{r})^T}{\sum_{t} \lambda^{T-t}}$$

            |$\lambda$|Interprétation|
            |---|---|
            | $\lambda \to 1$ | All observations carry the same weight. |
            | $\lambda \to 0$ | Only recent observations matter |
            | $\lambda = 0.94$ | Standard RiskMetrics JP Morgan |

        - Ledoit-Wolf methods (shrinkage) $$\Sigma_{LW} = (1 - \alpha) \cdot \Sigma_{sample} + \alpha \cdot F$$ 
            ```
                Σ_sample  →  Noisy but faithful to the data
                F         →  structured, stable but biased target
                α         →  intensity of shrinkage (optimal, calculated analytically)
            ```
            > Ledoit & Wolf (2004) find $\alpha$ **optimal analytically** → no calibration needed
            1. ***Ledoit-Wolf with constant variance*** (<span style="color:green"> simple, stable </span> but <span style="color:red"> with the strong hypothesis **all variances are equal** </span>), $F$ which is a matrix with constant variance $$F_{ij} = \begin{cases} \bar{\sigma}^2 & \text{if } i = j \\ 0 & \text{if } i \neq j \end{cases}$$
            2. ***Ledoit-Wolf single factor*** uses a factor method to value $F$ (like CAPM) (<span style="color:green"> economically motivated, good compromise bias/variance </span> but <span style="color:red"> assume that a factor alone explain the correlations</span>) $$F = \beta \beta^T \sigma_m^2 + D$$
            where $D$ is a diagonal matrix with specific variances
            3. ***Ledoit-Wolf with constant correlation*** (<span style="color:green"> keeps individual true variances, better compromise of the LW family </span> but <span style="color:red"> assume a constant correlation, which is generally incorrect </span>), $F$ is assuming an identical correlation between all the assets $$F_{ij} = \begin{cases} \sigma_i^2 & \text{if } i = j \\ \bar{\rho} \cdot \sigma_i \sigma_j & \text{if } i \neq j \end{cases}$$
        - Oracle Approximating (<span style="color:green"> Theoritically optimal </span> but <span style="color:red"> complex and computationally costful </span>):
            - Method of **Chen, Wiesel, Eldar & Hero (2010)** :
                - Find the matrix that minimizes the **expected quadratic loss**
                - "Oracle" because it approximates what a perfect estimator would do

- The investor views that depends on :
    - $P$ : Matrix representing the assets on which the investor has an opinion, each line is an opinion :
        - The sum of the row elements equals to 1 if it is an absolute view (*Asset 1 will have a performance of $x\%$*)
        - The sum of the row elements equals to 0 if it is a relative view (*Asset 1 will overperform Asset 2 and Asset 3 by $y\%$*)
        - Result $$P = \begin{pmatrix} 1 & 0 & 0 \\ 1 & -0.5 & -0.5 \end{pmatrix}$$
    - $Q$ : Is the matrix containing the values of these performances, in our exemple it would be $$Q = \begin{pmatrix} x\% \\ y\% \end{pmatrix}$$
    - $\Omega$ : Is the confidence matrix $$\Omega = \begin{pmatrix} \omega_1 & 0 \\ 0 & \omega_2 \end{pmatrix}$$ with $$\begin{aligned} &\omega \rightarrow 0 : \text{Total confidence in the view} \\ &\omega \rightarrow \infty : \text{View ignored}\end{aligned}$$
        - Calibrated using one of the following methods :
            - **Black-Litterman Method** : $\omega_i = \tau \cdot P_i \Sigma P_i^T$
            - **He & Litterman Method** : $\omega_i = \frac {1-c} {c} \cdot \tau \cdot P_i \Sigma P_i^T \text{ with } c \in [0,1]$
            - **Regression Method (Our Approach)** : $\omega_i = \text{Var}(\epsilon_i)$ (MSE of the macro-economic regression)


## Views
Now that we have calculated the prior and the covariance matrix, we will work on getting the investor views, for that, we wil implement different methods

### Momentum method
The main idea is "Assets that have performed well recently will continue to outperform"
This idea was documented by **Jegadeesh & Titman (1993)**, that shows that buying winners and selling losers on 3-12 months generates abnormal returns \
The theory behind
|Phenomenon|Description|
|---|---|
|Underreaction|Investors integrates slowly new informations|
|Herding|Investors follow trends|
|Disposition effect|Winners are sold too early, losers too late|

Classic signal : $$Mom_i = \frac {P_{t-1}}{P_{t-12}} - 1$$ Which is the momentum at time $i$ for the asset
>We exclude the last month (t-1) to avoid short-term reversal

To get our $Q_i$ :
1. We calculate the momentum of the asset : $$Mom_i = r_i(t-12, t-1) = \frac {P_{t-1}}{P_{t-12}} - 1$$ with $r_i$ return of asset $i$ (between month $t-1$ and month $t-12$)
2. We normalize the signal, to get the outperformers and underperformers. With the formula $$z_i = \frac {Mom_i-Mom}{\sigma_{Mom}}$$ with $Mom$ the mean of all the momentums, and $\sigma_{Mom}$ the standard deviation
3. We use the z-score obtained to calculate the view $Q_i$ : 
    1. First approach : $$Q_i = z_i * k$$ with k a coefficient
        - At first we will take $k = 0.05$
        - We will then calculate it as $k = \sigma_i \times \phi$ with $\phi$ the sharpe ratio wanted
        - Then we will use the information ratio approach (Grinold) with $k = \sigma_{residual} \times IC$ ($IC$ being the historical correlation between momentum scores and futur returns)
    2. Second approach (with only relative views) : 
        - Each line sum is 0 ($\sum p_i = 0$)
        - We select n top (with $p_i = \frac 1n$) and n bottom (with $p_i = - \frac 1n$) assets (performance wise)
        - We calculate $Q_i = (Z_{top} - Z_{bottom}) \times k$ (with Z_{top/bottom} the mean of Z-score of the top/bottom assets)

To get our $\Omega$ :
1. For the first approach, we will use the Black-Litterman method seen before  $\omega_i = \tau \cdot P_i \Sigma P_i^T$
2. For the second approach, we will use the historical variance of the difference between top and bottom assets returns $\omega_i = Var(R_{top} - R_{bottom})$

In [ ]:
def momentum(tickers, marketPrior, sigma, start_date="2015-01-01", k=0.05, tau=0.05):
    prices = getPrices(tickers, start_date).dropna()
    momentum = prices.iloc[-20] / prices.iloc[0] - 1
    z_scores = (momentum - momentum.mean()) / momentum.std()
    Q = marketPrior + z_scores.values * k

    P = np.zeros((len(tickers), len(tickers)))
    for i in range(len(tickers)):
        P[i, i] = 1
    Omega = tau * np.matmul(np.matmul(P, sigma), P.T)

    return P, Q, Omega

### Linear Regression

### Multiple Linear Regression

### Factors method (Fama-French)

### Multiple Linear Regression for Return Prediction

In this section, we employ a multiple linear regression model to forecast asset returns based on macroeconomic factors. This approach allows us to generate data-driven *views* on expected returns, which will be incorporated into the Black–Litterman framework.

For each asset $i$, we model the expected returns as:

$$
R_{i,t+1} = \alpha_i + \beta_{i,1} F^1_t + \beta_{i,2} F^2_t + \dots + \beta_{i,k} F^k_t + \varepsilon_{i,t+1}
$$

where:

- $R_{i,t+1}$ is the return of asset $i$ at time $t+1$,
- $F^j_t$ are the observed macroeconomic factors at time $t$,
- $\beta_{i,j}$ measure the sensitivity (exposure) of asset $i$ to factor $j$,
- $\alpha_i$ is the intercept (asset-specific alpha),
- $\varepsilon_{i,t+1}$ is an idiosyncratic error term assumed to be normally distributed.

The predicted returns ($\hat{R}_{i,t+1}$) from this model serve as absolute views ($Q$) in Black–Litterman, enhancing the equilibrium priors with empirical insights. We fit separate regressions for each asset to capture idiosyncratic sensitivities.

### Macro-Factor Selection

To forecast asset returns effectively, we select a parsimonious set of macroeconomic factors that are economically motivated, empirically validated, and publicly available. The goal is to capture key drivers of financial markets while minimizing model complexity, overfitting, and multicollinearity.

After analysis, we restrict the model to **four key macro factors**, each representing a distinct economic dimension. This reduction (from an initial consideration of five) avoids potential multicollinearity issues—e.g., between changes in long-term yields and yield curve slope—ensuring more stable and interpretable coefficients. Factors are sourced from reliable public databases like FRED (Federal Reserve Economic Data) and processed (e.g., differenced for stationarity where needed).

#### Selected Factors

1. **$\Delta$ 10Y Treasury Yield**  
   Measures changes in the 10-year U.S. Treasury yield, capturing shifts in long-term discount rates and valuation effects. This factor is particularly relevant for equity markets and growth stocks, as rising yields can compress multiples (e.g., as in Cochrane, 2008, on equity risk premiums).  
   Source: FRED (`GS10`); computed as first differences for stationarity.

2. **Inflation Surprise (CPI)**  
   Quantifies unexpected inflation shocks (actual CPI minus consensus forecast), which influence monetary policy expectations and real returns. High surprises can erode purchasing power and trigger rate hikes, impacting asset classes differently (e.g., Ang et al., 2008, on inflation hedging).  
   Source: FRED (`CPIAUCSL`) with approximations for surprises via lags or external forecasts.

3. **$\Delta$ VIX (Implied Volatility Index)**  
   Tracks changes in the CBOE Volatility Index, serving as a proxy for market risk aversion and uncertainty. Spikes in VIX often signal stress periods with negative equity returns (e.g., Carr & Wu, 2009, on volatility risk premium).  
   Source: FRED (`VIXCLS`); differenced to focus on shocks rather than levels.

4. **PMI Composite Index**  
   A forward-looking gauge of economic activity, blending manufacturing and services sectors. Higher PMI signals growth momentum, positively correlating with corporate earnings and stock returns (e.g., widely used in business cycle models by firms like JPMorgan).  
   Source: FRED (`NAPM` for ISM PMI) or equivalents; $z$-scored for normalization.

#### Why These Factors?

- **Economic Interpretation**: Each factor maps to a unique channel—interest rates (valuation), inflation (policy shocks), volatility (risk sentiment), and activity (growth)—providing clear insights into return drivers.

- **Distinctiveness and Robustness**: They are relatively orthogonal (low pairwise correlations, verified via heatmap in code), reducing multicollinearity risks. $VIF$ (Variance Inflation Factor) checks confirm stability.

- **Empirical Support**: Grounded in academic and practitioner literature (e.g., Fama–French extensions with macro factors; AQR’s macro timing models).

- **Practicality**: All are freely accessible, updated frequently, and suitable for a quantitative project at student level. The limited set (4 factors) ensures model parsimony, with better out-of-sample performance than overparameterized alternatives.

- **Rationale for Reduction**: An initial set of five included yield curve slope, but it was dropped due to moderate correlation (around $0.5$) with $\Delta$ 10Y Yield, prioritizing interpretability and avoiding inflated variances in estimates.

This factor framework forms a robust basis for conditional expected returns, bridging macroeconomics and portfolio optimization.

In [ ]:
def get_cleaned_data(ticker, start_date="2015-01-01"):
    end_date = datetime.today().strftime('%Y-%m-%d')
    
    # ---------------------------------------------------------
    # 1. MACRO Data (FRED)
    # ---------------------------------------------------------
    macro_setup = {
        'GS10': '10Y_Yield',    # Monthly
        'CPIAUCSL': 'CPI',      # Monthly
        'VIXCLS': 'VIX',        # Daily
        'INDPRO': 'IP'          # Monthly
    }
    
    macro_list = []
    for code, name in macro_setup.items():
        s = web.DataReader(code, 'fred', start_date, end_date)
        s.columns = [name]
        macro_list.append(s)
    
    macro_df = pd.concat(macro_list, axis=1).ffill()
    # ---------------------------------------------------------
    # 2. Data Stationarization
    # ---------------------------------------------------------
    macro_monthly_raw = macro_df.resample('ME').last()
    
    macro_st = pd.DataFrame(index=macro_monthly_raw.index)
    macro_st['10Y_Diff'] = macro_monthly_raw['10Y_Yield'].diff()
    macro_st['VIX_Diff'] = macro_monthly_raw['VIX'].diff()
    macro_st['CPI_Ret']  = macro_monthly_raw['CPI'].pct_change()
    macro_st['IP_Ret']   = macro_monthly_raw['IP'].pct_change()
    
    macro_monthly = macro_st.dropna()
 
    # ---------------------------------------------------------
    # 3. Asset Return (YFinance)
    # ---------------------------------------------------------
    asset_raw = yf.download(ticker, start=start_date, end=end_date, progress=False)
    
    if isinstance(asset_raw.columns, pd.MultiIndex):
        prices = asset_raw['Close', ticker]
    else:
        prices = asset_raw['Close']
        
    # Timezone Harmonization (UTC/EST)
    if prices.index.tz is not None:
        prices.index = prices.index.tz_localize(None)

    # Monthly compounded return ('ME')
    asset_monthly_ret = prices.pct_change().resample('ME').apply(lambda x: (1 + x).prod() - 1)
    asset_monthly_ret.name = 'Asset_Return'

    # ---------------------------------------------------------
    # 4. Final DataFrame
    # ---------------------------------------------------------
    final_df = pd.concat([asset_monthly_ret, macro_monthly], axis=1).dropna()
    
    # X (Features)
    X = final_df.drop(columns=['Asset_Return']).shift(1).dropna()
    
    # y (Target)
    y = final_df['Asset_Return'].loc[X.index]
    
    # ---------------------------------------------------------
    # 5. NORMALIZATION (Z-Score)
    # ---------------------------------------------------------
    X_norm = (X - X.mean()) / X.std()
    
    return X_norm, y

# ==========================================
# Test
# ==========================================
try:
    X, y = get_cleaned_data('SPY')
    print("✅ Succes !")
    print(f"X dimension : {X.shape}")
    print(f"y dimension : {y.shape}\n")
    print("Overview of the first 5 lines of X (Normalized macro factors):")
    print(X.head())
    print("\nOverview of the first 5 lines of y (SPY Returns) :")
    print(y.head())
except Exception as e:
    print(f"❌ Error during execution : {e}")

In [ ]:
def getViews(tickers, marketPrior, sigma, method="momentum", k=0.05, tau=0.05, start_date="2015-01-01",):
    match method:
        case "momentum":
            print("momentum")
            P, Q, Omega = momentum(tickers, marketPrior=marketPrior, start_date=start_date, sigma=sigma, k=k, tau=tau)
        case "Fama-French":
            print("Fama-French")
            P, Q, Omega = fama_french() 
        case "multi_linear_regression":
            print("multi_linear_regression")
            P, Q, Omega = multi_linear_regression()

    return P, Q, Omega

In [ ]:
def  Black_Litterman_Opt(tickers, market="SPY", cov_method='constant_correlation', risk_free_rate=0.02, start_date="2015-01-01") :
    prices = getPrices(tickers, start_date)
    marketPrices = getPrices(market, start_date)
    weights, marketCaps = DisplayMarketCaps(tickers)
    #Calculating Sigma
    match cov_method : 
        case "sample_cov":
            print("We will use a sample_cov")
            sigma = risk_models.risk_matrix(prices,method="sample_cov")
        case "semicovariance":
            print("We will use semicovariance")
            sigma = risk_models.risk_matrix(prices,method="semicovariance")
        case "exp_cov":
            print("We will use the exponential covariance")
            sigma = risk_models.risk_matrix(prices,method="exp_cov")
        case "ledoit_wolf":
            print("We will use ledoit_wolf")
            sigma = risk_models.risk_matrix(prices,method="sampleledoit_wolf_cov")
        case "ledoit_wolf_constant_variance":
            print("We will use ledoit_wolf_constant_variance")
            sigma = risk_models.risk_matrix(prices,method="ledoit_wolf_constant_variance")
        case "ledoit_wolf_single_factor":
            print("We will use ledoit_wolf_single_factor")
            sigma = risk_models.risk_matrix(prices,method="ledoit_wolf_single_factor")
        case "ledoit_wolf_constant_correlation":
            print("We will use ledoit_wolf_constant_correlation")
            sigma = risk_models.risk_matrix(prices,method="ledoit_wolf_constant_correlation")
        case "oracle_approximating":
            print("We will use oracle_approximating")
            sigma = risk_models.risk_matrix(prices,method="oracle_approximating")

    #Calculating delta
    delta = black_litterman.market_implied_risk_aversion(marketPrices)
    #Calculating Prior
    marketPrior = black_litterman.market_implied_prior_returns(marketCaps, delta, sigma)

    #Adding views
    P, Q, Omega = getViews(tickers, marketPrior, sigma, method="momentum", k=0.05, tau=0.05)

    ## Calculating posterior
    bl = BlackLittermanModel(sigma, market_prior=marketPrior, P=P, Q=Q, Omega=Omega)
    posterior_returns = bl.bl_returns()
    posterior_cov = bl.bl_cov()

    ## Applying mean-variance optimization
    ef = EfficientFrontier(posterior_returns, posterior_cov)
    weights = ef.max_sharpe(risk_free_rate=risk_free_rate)
    cleaned_weights = ef.clean_weights()
    
    #plotting
    plotting.plot_covariance(sigma, plot_correlation=True)
    marketPrior.plot.barh(figsize=(10,5))
    delta